## (Challenge) Business Question #3: Talent Supply & Demand Baseline Assessment

**Datasets Used:** `talent_demand_pay`, `company_num_postings`, `top_employers`

### Scenario

Your consulting firm has been engaged by **TechVentures Capital**, a venture capital firm that invests heavily in AI and data-driven startups. They're preparing their 2026 investment thesis and need to understand the competitive landscape for data talent in the United States.

The Managing Partner has specifically asked: _"We keep hearing that it's nearly impossible to hire Data Scientists and Data Engineers. Before we advise our portfolio companies on hiring strategies, we need hard numbers on the actual state of the market."_

### Business Questions to Answer

1. **Market Overview**: What is the current state of talent supply vs. demand for both roles? Calculate the talent-to-demand ratio (how many available professionals per open job posting).
2. **Hiring Competition Intensity**: How many companies are actively competing for this talent? Provide total unique employer counts for each role.
3. **Compensation & Difficulty Context**: What is the median base pay for each role, and how does this correlate with the hiring difficulty index?

### Deliverable Required

**Executive Brief (Markdown Report)** containing:

- A summary table comparing both roles across key metrics (supply, demand, ratios, pay, difficulty)
- 2-3 paragraph interpretation of what these numbers mean for their portfolio companies
- One clear recommendation regarding which role poses greater hiring risk

**Presentation Tip:** Your audience is non-technical investors. Keep SQL jargon out of the deliverable, focus on business insights.

In [0]:
--Use source table for talent_demand_pay to get relative interpretation of metrics for data scientist and data engineer
-- Convert foreign currencies to USD
SELECT 
  job_role,
  country,
  talent_size,
  talent_demand,
  ROUND((talent_size/talent_demand), 2) AS supply_demand_ratio,
  ROUND((median_base_pay), 2) AS median_base_pay, 
  CASE
    WHEN country = 'Brazil' THEN ROUND((median_base_pay * 0.19), 2)
    WHEN country = 'India' THEN ROUND((median_base_pay * 0.12), 2)
    ELSE ROUND((median_base_pay), 2)
  END AS usd_median_base_pay,
  hiring_difficulty_index
FROM draup_inc_global_labor_market_data_talent_intelligence_sample.role_country.talent_metrics_country_job_role
ORDER BY 
  supply_demand_ratio ASC; 


job_role,country,talent_size,talent_demand,supply_demand_ratio,median_base_pay,usd_median_base_pay,hiring_difficulty_index
Data Scientist,Brazil,11875.0,606,19.6,216448.94,41125.3,5.6
Data Engineer,United States of America,218760.0,10455,20.92,130622.67,130622.67,5.8
Data Scientist,United States of America,134300.0,5753,23.34,144000.0,144000.0,5.9
Data Engineer,Brazil,24500.0,1000,24.5,182237.84,34625.19,5.1
Data Engineer,India,156730.0,5304,29.55,1085696.0,130283.52,5.0
Financial Analyst,Brazil,98842.0,3319,29.78,126554.51,24045.36,6.0
Product Manager,United States of America,463290.0,15499,29.89,130834.0,130834.0,5.9
Financial Analyst,United States of America,424675.0,13126,32.35,88886.46,88886.46,5.7
Product Manager,Brazil,35090.0,1015,34.57,148481.28,28211.44,5.5
Data Scientist,India,96504.0,2187,44.13,1231784.65,147814.16,4.9


In [0]:
-- Confirm distinct countries for conversion
SELECT 
  DISTINCT country
FROM 
  draup_inc_global_labor_market_data_talent_intelligence_sample.role_country.talent_metrics_country_job_role

country
Brazil
India
United States of America


### How to reconcile the above demand metrics with the below posting numbers??

In [0]:
USE CATALOG challenges;
USE SCHEMA challenge_tables;
--company_num_postings is already filtered for USA and Data Roles
SELECT 
  job_role, 
  SUM(job_postings_count) AS total_posts
FROM 
  company_num_postings
GROUP BY 
  job_role;

job_role,total_posts
Data Scientist,118142
Data Engineer,211108


In [0]:
USE CATALOG challenges;
USE SCHEMA challenge_tables;
--group job_role and count unique company names hiring for each role
SELECT 
  job_role,
  country, 
  COUNT(DISTINCT company_name) AS total_unique_companies_hiring
FROM 
  company_num_postings
GROUP BY 
  job_role,
  country;

job_role,country,total_unique_companies_hiring
Data Scientist,United States of America,16997
Data Engineer,United States of America,27354


In [0]:
-- Glimpse of top 10 Companies 
USE CATALOG challenges;
USE SCHEMA challenge_tables;

SELECT 
  *  
FROM 
  company_num_postings
ORDER BY
  job_postings_count DESC
LIMIT 10;


job_role,country,company_name,job_postings_count
Data Engineer,United States of America,Jobs via Dice,5916
Data Engineer,United States of America,Tietalent,3648
Data Engineer,United States of America,Amazon.com,3201
Data Engineer,United States of America,Oracle Corporation,2876
Data Scientist,United States of America,Tietalent,2416
Data Engineer,United States of America,Canonical Ltd.,2241
Data Engineer,United States of America,Capital One Financial Corporation,2223
Data Scientist,United States of America,SynergisticIT,2103
Data Engineer,United States of America,"Amazon Web Services, Inc.",1940
Data Scientist,United States of America,Capital One Financial Corporation,1882


In [0]:
--Combined table for aggregations, included correlation in table 1, atm
USE CATALOG challenges;
USE SCHEMA challenge_tables;

WITH all_talent_metrics AS (
SELECT 
  job_role,
  country,
  talent_size,
  talent_demand,
  ROUND((talent_size/talent_demand), 2) AS supply_demand_ratio,
  ROUND((median_base_pay), 2) AS median_base_pay, 
  hiring_difficulty_index,
  ROUND(CORR(median_base_pay, hiring_difficulty_index) OVER(), 2) AS pay_hiring_diff_corr
FROM draup_inc_global_labor_market_data_talent_intelligence_sample.role_country.talent_metrics_country_job_role
ORDER BY 
  supply_demand_ratio ASC),

total_job_posts AS (
  SELECT 
  job_role,
  country, 
  SUM(job_postings_count) AS total_posts
FROM 
  company_num_postings
GROUP BY 
  job_role,
  country
),
unique_companies_hiring AS (
  SELECT 
  job_role,
  country, 
  COUNT(DISTINCT company_name) AS total_unique_companies_hiring
FROM 
  company_num_postings
GROUP BY 
  job_role,
  country
)
--make sure atm country matches tjp and uch because those are already filtered for USA and Data roles
SELECT 
  atm.job_role,
  atm.country,
  atm.talent_size,
  tjp.total_posts,
  ROUND((atm.talent_size/tjp.total_posts), 2) AS size_to_posts_ratio,
  uch.total_unique_companies_hiring, 
  atm.supply_demand_ratio,
  atm.median_base_pay,
  atm.hiring_difficulty_index,
  atm.pay_hiring_diff_corr
FROM 
  all_talent_metrics atm 
JOIN 
  total_job_posts tjp
  ON atm.job_role = tjp.job_role
  AND atm.country = tjp.country
JOIN 
  unique_companies_hiring uch
  ON atm.job_role = uch.job_role
  AND atm.country = uch.country;




job_role,country,talent_size,total_posts,size_to_posts_ratio,total_unique_companies_hiring,supply_demand_ratio,median_base_pay,hiring_difficulty_index,pay_hiring_diff_corr
Data Scientist,United States of America,134300.0,118142,1.14,16997,23.34,144000.0,5.9,-0.52
Data Engineer,United States of America,218760.0,211108,1.04,27354,20.92,130622.67,5.8,-0.52


#### Pivoting the Above Results

In [0]:
--Combined table of aggregations, included correlation in table 1, atm
USE CATALOG challenges;
USE SCHEMA challenge_tables;

-- had to break all_talent_metrics due to lateral CASE and CORR() statements limitation
WITH base_talent_metrics AS (
  SELECT 
  job_role,
  country,
  talent_size,
  talent_demand,
  ROUND((talent_size/talent_demand), 2) AS size_demand_ratio,
  ROUND((median_base_pay), 2) AS median_base_pay, 
  CASE
    WHEN country = 'Brazil' THEN ROUND((median_base_pay * 0.19))
    WHEN country = 'India' THEN ROUND((median_base_pay * 0.12))
    ELSE ROUND((median_base_pay), 2)
  END AS usd_median_base_pay,
  hiring_difficulty_index
FROM 
  draup_inc_global_labor_market_data_talent_intelligence_sample.role_country.talent_metrics_country_job_role

),
all_talent_metrics AS (
SELECT
    *,
    ROUND(CORR(usd_median_base_pay, hiring_difficulty_index) OVER (), 2) AS pay_hiring_diff_corr
  FROM base_talent_metrics),
total_job_posts AS (
  SELECT 
  job_role,
  country, 
  SUM(job_postings_count) AS total_posts
FROM 
  company_num_postings
GROUP BY 
  job_role,
  country
),
unique_companies_hiring AS (
  SELECT 
  job_role,
  country, 
  COUNT(DISTINCT company_name) AS total_unique_companies_hiring
FROM 
  company_num_postings
GROUP BY 
  job_role,
  country
),
--make sure atm country matches tjp and uch because those are already filtered for USA and Data roles
combined_data AS (
  SELECT 
  atm.job_role,
  atm.country,
  atm.talent_size,
  tjp.total_posts,
  atm.talent_demand,
  ROUND((atm.talent_size/tjp.total_posts), 2) AS size_to_posts_ratio,
  atm.size_demand_ratio,
  uch.total_unique_companies_hiring, 
  atm.median_base_pay,
  atm.hiring_difficulty_index,
  atm.pay_hiring_diff_corr
FROM 
  all_talent_metrics atm 
JOIN 
  total_job_posts tjp
  ON atm.job_role = tjp.job_role
  AND atm.country = tjp.country
JOIN 
  unique_companies_hiring uch
  ON atm.job_role = uch.job_role
  AND atm.country = uch.country
)
SELECT *
-- Unstack the first two values' statements to understand the nested FROM statement
FROM (
  SELECT 
    job_role,
    'talent_size' AS metric, 
    talent_size AS value 
  FROM combined_data
  UNION ALL 
  SELECT 
    job_role, 
    'total_posts', total_posts 
  FROM 
    combined_data
  UNION ALL SELECT job_role, 'talent_demand', talent_demand FROM combined_data   
  UNION ALL SELECT job_role, 'size_to_posts_ratio', size_to_posts_ratio FROM combined_data
  UNION ALL SELECT job_role, 'size_demand_ratio', size_demand_ratio FROM combined_data
  UNION ALL SELECT job_role, 'total_unique_companies_hiring', total_unique_companies_hiring FROM combined_data
  UNION ALL SELECT job_role, 'median_base_pay', median_base_pay FROM combined_data
  UNION ALL SELECT job_role, 'hiring_difficulty_index', hiring_difficulty_index FROM combined_data
  UNION ALL SELECT job_role, 'pay_hiring_diff_corr', pay_hiring_diff_corr FROM combined_data
)
PIVOT (
  MAX(value)
  FOR job_role IN ('Data Scientist', 'Data Engineer')
);



metric,Data Scientist,Data Engineer
talent_size,134300.0,218760.0
total_posts,118142.0,211108.0
talent_demand,5753.0,10455.0
size_to_posts_ratio,1.14,1.04
size_demand_ratio,23.34,20.92
total_unique_companies_hiring,16997.0,27354.0
median_base_pay,144000.0,130622.67
hiring_difficulty_index,5.9,5.8
pay_hiring_diff_corr,-0.1,-0.1


### Find the Confidence Interval for Correlation Results

> 1. Transform/stretch the correlation coefficient using FisherZ into normal distribution space
> 2. Find the 95% CI bounds for the this new z-score
> 3. Compress the z-score back with EXP(), e^2z + 1 / e^2z - 1

In [0]:
USE CATALOG challenges;
USE SCHEMA challenge_tables;
-- Make sure currency is all in USD for standardization 
--Not a statistically significant enough correlation to make conclusions, based on confidence interval of 95% 
WITH usd_added_values AS (
  SELECT 
    CASE
      WHEN country = 'Brazil' THEN ROUND((median_base_pay * 0.19), 2)
      WHEN country = 'India' THEN ROUND((median_base_pay * 0.12), 2)
      ELSE median_base_pay
    END AS usd_median_base_pay,
  hiring_difficulty_index
  FROM 
    draup_inc_global_labor_market_data_talent_intelligence_sample.role_country.talent_metrics_country_job_role
),
calc_stats AS(
  SELECT 
    CORR(usd_median_base_pay, hiring_difficulty_index) AS r,
    COUNT(*) AS n
  FROM 
    usd_added_values
  WHERE 
    usd_median_base_pay IS NOT NULL
    AND hiring_difficulty_index IS NOT NULL 
),
FisherZ AS (
  SELECT 
    r, n, 
    0.5 * LN((1 + r) / (1 - r)) AS z,
    1 / SQRT(n - 3) AS se
  FROM 
    calc_stats
),
Bounds AS (
  SELECT 
    r, n, 
    z - (1.96 * se) AS lower_bound,
    z + (1.96 * se) AS upper_bound
  FROM 
    FisherZ 
)
SELECT 
  ROUND(r, 4) AS correlation,
  n AS sample_size,
  ROUND(((EXP(2 * lower_bound) - 1) / (EXP(2 * lower_bound) + 1)), 4) AS conf_lower_95,
  ROUND(((EXP(2 * upper_bound) - 1) / (EXP(2 * upper_bound) + 1)), 4) AS conf_upper_95
FROM 
  Bounds; 

correlation,sample_size,conf_lower_95,conf_upper_95
-0.0982,15,-0.5812,0.436
